# 🫀 퀘스트 46 · Q7-R — **자를 고치고, 결정 가능한 질문으로 갈아탄다**

| | **MedKOS / `notebooks/quest46_q7r_calibrate.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0066`(Q7-Q) · `ailab-2026-0065`(Q7-P) |
| 규약 | **R22 · R26 ② · R27 ② ③ · R29 ① ② · R30 ① · R31 ④ ⑤ · R32 ① ② ④ ⑤ ⑥ · R33 ① ② ③ ④ ⑤** |
| 학습 | **0회**(전역 로지스틱만) · GPU 불필요 |

## Q7-Q 가 남긴 상태 — 병목은 창이 아니라 자다

```
관측 창 고유항        +0.03        (= R 파 진폭의 **0.8%**)
관문 CI 반폭          0.117        ← 자의 눈금이 재려는 것보다 **4배 굵다**
등가 필요 개체        1761 · 2000 · 43823
```

그리고 **음성 대조가 누출만 나른다**는 게 실측으로 확인됐다:

```
stt_22 초과 +0.1216  ≈  누출 바닥 +0.1204 (f2_5)     차이 +0.0012
⇒ 모든 창의 초과 = 공통 누출(≈0.12) + 창 고유항
   p_mid_22 +0.031 · p_late_strict +0.031 · stt_22 **+0.001** · p_early_22 −0.046
```

**Q1 이 실패한 건 P 창이 무너져서가 아니라 음성 대조가 안 죽어서다.**

## 그래서 넷을 한다

| | 무엇 | 왜 |
|---|---|---|
| **R1 ★★** | **주 관문 교체** — 교차환자 리듬 전용 대비 **ΔAUPRC** 등가(±0.01) | 팔 간 등가는 필요 n 1761 로 원리적 불가(R33 ⑤). 누출이 두 모형에 공통으로 들어가 **상쇄**된다 |
| **R2 ★★** | **관문 통과형 양성 대조** — `a ∈ {0.005 … 0.05}` · **개체별 이질 파형** · 관문에 그대로 통과 | Q7-Q 는 **수준 추정기**만 검정하고 관문의 검출력을 주장했다(R33 ①). 산출물 = **관문 MDE** |
| **R3 ★★** | **추정량을 실측으로 고른다** — `raw`/`lin`/`rank`/`strat`/**`dr`** 를 (회수량 × CI 폭)으로 | R31 ③ vs R32 ⑥ 충돌을 규칙이 아니라 **재서** 끝낸다(R33 ③) |
| **R4 ★** | **타이밍 vs 형태 분해** + **지터 대조** | Q7-Q 의 적응형 −0.0666 이 (A)타이밍인지 (B)검출기 잡음인지 |

보조로 **R5** 층 키 교체(`f2_16` → `f2_5`)로 누출 바닥 모형을 확증한다.

## ⚠️ 「AIPW」를 그대로 쓰지 않는 이유 — 먼저 밝혀둔다

AIPW/이중강건은 **인과 추정량**(처치의 평균효과)에 정의된다. 지금 재는 건
**짝지은 매크로 AUROC 차이**라 처치도 결과모형도 정의돼 있지 않다. **그대로 AIPW 라고
부르면 과장이다.**

대신 **구현 가능한 이중강건 *스타일*** 합성을 넣는다.

```
dr = strat_auc( residualize(점수, **확장 기저**), key = f1 × f2_16 )
     확장 기저 = 기본 기저 + **누출 채널 f2_{5,10,20}** + f4 + trend
```

- **잔차화가 맞으면** 일관 — 누출 채널을 직접 지운다
- **층화가 맞으면** 일관 — 함수형이 틀려도 칸 안 비교는 산다
- 프로브는 **기저 밖 새 `k`(7·14·28)** 로 둔다 — 안 그러면 검증이 안 된다

★ **진짜 AIPW 는 R1 쪽에서 자연스럽다** — 비트 수준 전역 모형은 추정량이 스칼라라
영향함수 기반 추정이 정의된다. 이번엔 **R1 을 그 길로 열어두는 것**까지만 한다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **R1 ★★ (주)** | 교차환자(LORO) 전역 모형 · 리듬 전용 대비 **창 특징 추가**의 **ΔAUPRC** | **등가 ±0.01** + 우월성 **둘 다**(R31 ①) |
| **R1b ★** | 같은 비교의 **고특이도 부분 AUC**(특이도 0.95–1.0) | 등가 ±0.01 + 우월성 |
| **R2 ★★** | (관문 아님 · **교정**) 주입 `a` 를 **R1b 판형 짝 관문**에 통과시켜 **MDE** | 최소 한 점이 **미검출**이어야 유효(R33 ②) |

⚠️ **천장효과를 먼저 없앤다** — Q7-Q 에서 `lr_rhy` 무잔차 AUROC 가 **0.9600** 이었다.
그 위의 ΔAUROC 는 압축된다. 유병률 0.08 이므로 **AUPRC 와 동작점 부분 AUC 가 1차**이고
ΔAUROC 는 참고로만 낸다. **이 재측정 전에는 「형태 축 종료」를 확정하지 않는다.**

### 판정표 (R29 ②)

- **R1 등가 ✅ · R1b 등가 ✅** → **형태 축 종료.** 창 형태는 리듬 위에 임상적으로 의미
  있는 것을 못 얹는다. 퀘스트의 형태 갈래를 닫는다
- **R1 등가 ✅ · R1b 우월 ✅** → ★ **전역 순위는 안 바꾸는데 동작점은 바꾼다.**
  이게 가장 흥미로운 결과이고 `ecg-oppoint` 트랙으로 넘어간다
- **어느 쪽이든 MDE 위에서 미결** → 표본을 늘린다(**DB 풀링** · R3 가 필요 n 을 준다)
- **MDE 아래 미결** → ⛔ **측정 한계**. 어떤 결론 분기도 안 탄다

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def equiv(lo, hi, margin):
    if lo > -margin and hi < margin:
        return "✅ 등가"
    if lo > margin or hi < -margin:
        return "❌ 차이 있음"
    return "⚠️ 미결"

def need_n(n, lo, hi, mean, margin):
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    if slack <= 0:
        return None
    return float(n) * (((hi - lo) / 2.0) / slack) ** 2

def mde(lo, hi):
    """★ **최소 검출 효과** = CI 반폭(R33 ①). 관측 효과가 이 아래면 측정 한계다."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def judge(mean, lo, hi, n, margin):
    """등가·우월성·필요표본·**MDE** 를 한 관문에서 전부(R31 ① · R30 ① · R33 ①)."""
    eq, sup = equiv(lo, hi, margin), decide(lo, hi, 0.0, ">")
    nn, m_ = need_n(n, lo, hi, mean, margin), mde(lo, hi)
    if nn is None:
        frame = (f"⛔ **등가 판정 불가** — 점추정 {mean:+.4f} 이 여유 ±{margin} 밖 "
                 "→ **우월성 프레임**(R31 ①)")
    elif not np.isfinite(nn):
        frame = "⚠️ 필요 개체 계산 불가"
    else:
        frame = f"등가 필요 개체 ≈ **{nn:.0f}** (현재 {n})"
    limit = "⛔ **MDE 아래 — 측정 한계**" if abs(mean) < m_ else "▸ MDE 위"
    return eq, sup, nn, m_, f"{frame} · MDE **{m_:.4f}** · {limit}"

def ladder_read(lad, tol=1e-9):
    """★ `raw` 앵커로 교란을 판정한다(R32 ②).
    ★★ **단조성을 실제로 검사한다**(R33 ④) — Q7-Q 는 끝값/시작값 비만 보고 세 관문 모두
    「단조 감쇠」로 찍었는데 실제 사다리는 오르내렸다. 문구가 사실보다 강했다."""
    if "raw" not in lad:
        return "⛔ `raw` 앵커가 없다 — 교란 판정 불가(R32 ②)"
    ks = [k for k in lad if np.isfinite(lad[k])]
    if len(ks) < 3:
        return "⚠️ 사다리가 짧다"
    vals = [lad[k] for k in ks]
    amax = ks[int(np.argmax(np.abs(vals)))]
    total = lad[ks[-1]] - lad["raw"]
    mono = all(abs(vals[i]) >= abs(vals[i + 1]) - tol for i in range(len(vals) - 1))
    tail = f"실제 통제 효과 raw→{ks[-1]} **{total:+.4f}**"
    if amax != "raw":
        return (f"▸ **`raw` 가 최댓값이 아니다**(최대 `{amax}`) → 사다리 내부 변화를 "
                f"**교란으로 읽지 않는다**(중간 추정량 인공물). {tail}")
    if mono and abs(vals[-1]) <= 0.7 * abs(vals[0]):
        return f"⚠️ **`raw` 최댓값 + 단조 감쇠 → 잔여 교란**(R31 ③). {tail}"
    if abs(vals[-1]) <= 0.7 * abs(vals[0]):
        return (f"▸ `raw` 가 최댓값이고 끝값이 낮다 — **다만 단조가 아니다**(오르내림). "
                f"「단조 감쇠」라고 쓰지 않는다(R33 ④). {tail}")
    return f"▸ `raw` 가 최댓값이나 감쇠가 약하다. {tail}"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S, RPRE, NB_BOOT = 20260803, 1, 100, 4000

# ── 창 (Q7-Q 승계 — 폭 22 · 겹침 없음 · 여기서 새로 고르지 않는다)
W22 = 22
SEGS22 = {"p_early_22": (0, 22), "p_mid_22": (31, 53),
          "p_late_strict": (53, 75), "stt_22": (130, 152)}
NEG22 = "stt_22"
GATE_WIN = "p_mid_22"            # ★ Q7-Q 의 유일한 단서 — 주입·짝 관문의 대상

# ── ★★ R2 관문 통과형 양성 대조
AMPS = (0.005, 0.01, 0.02, 0.03, 0.05)   # ★ 최소점이 **미검출**이어야 유효(R33 ②)
HETERO = 0.35                            # ★ 개체별 파형 이질성(동일 파형 금지 · R33 ①)
QRS_LO, QRS_HI = 85, 115

# ── QRS 개시 · P 피크 (Q7-Q 승계)
ONSET_SEARCH, ONSET_FRAC, ONSET_RUN = (70, 96), 0.15, 3

# ── 기저 · 층 키
BASIS_K = (4, 6, 8, 12, 16, 24, 32)
LEAK_K  = (5, 10, 20)            # ★ `dr` 확장 기저에 **직접 넣는** 누출 채널
PROBE_K = (7, 14, 28)            # ★ 기저 **밖** 새 프로브 — 안 그러면 검증이 안 된다
HIST_K, LB_K, F2_BIN, TREND_W = 64, 16, 0.02, 8
KEY_ALT_K = 5                    # R5 — 층 키를 f2_16 → f2_5 로 갈아본다
MIN_S_TPL, K_FOLD, N_REPEAT, N_SHUF = 20, 5, 3, 20

# ★ R3 — 추정량을 **실측으로 고른다**(R33 ③). `dr` 는 이중강건 **스타일**이지 AIPW 가 아니다.
LADDER = ("raw", "lin", "rank", "strat", "dr")
PRIMARY = "strat"                # 이번엔 **잠정**. R3 가 다음 실행의 주 추정량을 정한다

# ── R1 교차환자 전역 모형
N_PCA = 5                        # 창 특징 = 창 PCA 상위 성분(**학습 레코드에서만 적합**)
SPEC_LO = 0.95                   # 동작점 부분 AUC 구간 (특이도 0.95–1.0)
EQ_DELTA = 0.01                  # ★ 사전등록 등가 임계 (ΔAUPRC · Δ부분AUC)
BONF2 = 0.05 / 2 / 2             # 1차 가족 {R1 · R1b}
EQUIV_MARGIN = EQ_DELTA

RULE_CHECK = {
    "R16 fallback 없음":        "자산 셀에서 예외 삼킴 없음",
    "R22 교차적합":             "개체 내 겹 밖 · 전역은 **LORO(개체 단위)**",
    "R26 ② 자기 null":          "라벨셔플 null 위 초과분 · SE 전파",
    "R27 ② 차이 우선":          "수준보다 짝의 차이를 1차로",
    "R27 ③ 폭 정합":            "폭 22 계열 전부 동일 · 겹침 없음(코드로 강제)",
    "R29 ② 측정 불가 분기 금지": "★ ⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R31 ① 등가/우월 둘 다":    "judge() 가 둘 다 + 필요표본 + **MDE**",
    "R31 ⑤ 잔차는 팔 간 비교 X": "잔차는 사다리에서만",
    "R32 ① 키가 이미 통제하나": "마스크 없음 — f1 정확 층화",
    "R32 ② raw 앵커":           "사다리에 raw 포함",
    "R32 ④ 누출 바닥 양수":     "양의 초과 max · 음의 초과는 분리",
    "R32 ⑤ 바닥은 수준에만":    "짝의 차이에서 안 뺀다",
    "R33 ① 관문 통과형 대조":   "★★ 주입을 **관문에 그대로 통과**시켜 MDE 를 낸다",
    "R33 ② 격자가 바닥을 감싼다": "★ a=0.005 에서 **미검출**이 나와야 유효",
    "R33 ③ 추정량은 실측 선택": "★ (회수량 × CI 폭) → MDE 최소",
    "R33 ④ 단조성 검사":        "★ ladder_read 가 단조성을 실제로 본다",
    "R33 ⑤ 리듬 대비 임계":     "★★ 주 관문을 **ΔAUPRC 등가 ±0.01** 로 교체 · 천장효과 제거",
}

CONFIG = dict(
    exp="quest46_q7r_calibrate", quest="ailab-2026-0046", step="svdb-calibrate",
    parent_exp=["quest46_q7q_late_strict", "ailab-2026-0066"],
    purpose=("Q7-Q 가 병목을 **창의 위치가 아니라 관문의 분해능**으로 특정했다 — 창 "
             "고유항 +0.03(R 진폭의 0.8%) vs 관문 CI 반폭 0.117. 그리고 **음성 대조가 "
             "누출만 나른다**(stt_22 +0.1216 ≈ 누출 바닥 +0.1204)는 게 확인됐다. "
             "그래서 창 질문을 보류하고 ① 주 관문을 **결정 가능한 것**(교차환자 리듬 "
             "대비 ΔAUPRC 등가)으로 갈아타고 ② 양성 대조를 **관문에 통과**시켜 MDE 를 "
             "실측하고 ③ 추정량을 규칙이 아니라 **재서** 고르고 ④ 적응형 창의 −0.0666 이 "
             "타이밍인지 검출기 잡음인지 가른다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    windows={k: list(v) for k, v in SEGS22.items()}, gate_win=GATE_WIN, neg=NEG22,
    amps=list(AMPS), hetero=HETERO, ladder=list(LADDER), primary=PRIMARY,
    basis_k=list(BASIS_K), leak_k=list(LEAK_K), probe_k=list(PROBE_K),
    n_pca=N_PCA, spec_lo=SPEC_LO, eq_delta=EQ_DELTA, rule_check=RULE_CHECK,
    predictions={
        "R1": f"★★ **주 관문** — 교차환자(LORO) 전역 모형에서 리듬 전용 대비 창 특징 "
              f"추가의 **ΔAUPRC**. 등가 ±{EQ_DELTA} + 우월성 **둘 다**. Bonferroni 2. "
              "⚠️ AUROC 가 아니라 AUPRC 인 이유는 **천장효과**다(Q7-Q lr_rhy raw 0.9600)",
        "R1b": f"★ 같은 비교의 **고특이도 부분 AUC**(특이도 {SPEC_LO}–1.0). "
               "전역 순위는 안 바꾸면서 **동작점 정밀도**는 올릴 수 있다",
        "R2": "★★ (교정) 주입 `a` 를 **짝 관문에 그대로 통과**시켜 **관문 MDE** 를 낸다"
              "(R33 ①). 격자 최소점 a=0.005 에서 **미검출**이 나와야 바닥을 감싼 것이다"
              "(R33 ②). 주입 파형은 **개체별로 다르게**(동일 파형은 상한만 잰다)",
        "R3": "★★ (선택) `raw`/`lin`/`rank`/`strat`/`dr` 를 같은 데이터에서 — "
              "**회수량(편향 대리) × CI 폭(분산) → MDE**. **MDE 최소**를 다음 실행의 "
              "주 추정량으로 사전등록한다(R33 ③)",
        "R4": "★ (분해) (a)고정 (b)개시정렬 (c)스칼라 PR 단독 (d)PR 잔차화 + "
              "**지터 주입 대조** — Q7-Q 의 −0.0666 이 (A)타이밍인지 (B)검출기 잡음인지",
        "R5": "(진단) 층 키를 `f2_16` → `f2_5` 로 갈면 `stt_22` 초과가 0 으로 가는가 "
              "— 누출 바닥 모형의 확증. **칸 수 타당성 먼저**"},
    caveat=("★ **`dr` 는 AIPW 가 아니다** — AIPW 는 인과 추정량에 정의되는데 짝지은 "
            "매크로 AUROC 차이엔 처치도 결과모형도 정의돼 있지 않다. `dr` 는 "
            "**잔차화(확장 기저) → 층화** 의 이중강건 **스타일** 합성이고, 그렇게만 "
            "부른다. 진짜 AIPW 는 R1 의 비트 수준 전역 모형 쪽에서 자연스럽다. "
            "★ **주입은 개체별 이질 파형**이라 Q7-Q 의 상한보다 현실적이지만 여전히 "
            "낙관적이다(주입은 라벨과 완전 상관). ★ **R1 의 창 특징은 PCA** 라 "
            "「무엇이 기여했나」를 말하지 않는다 — 「기여가 있나 없나」만 묻는다. "
            "★ 창 질문(P 종말부)은 **R3 가 추정량을 정한 뒤** 다시 연다. "
            "학습 0회(전역 로지스틱만) · GPU 불필요 · 예상 35~50분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7r_calibrate", CONFIG, project=PROJECT)
run.log(f"설정 ✅ 사다리 {' → '.join(LADDER)} · 주입 {AMPS} · 등가 임계 ±{EQ_DELTA}")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<24} {v_}")

In [ ]:
# CELL 2 — 【R-0a】 자산 · 매핑 (fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.ascontiguousarray(np.asarray(d5["beat"])[keep]).astype("float32")
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【R-0a】 자산 · 창")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개 · 유병률 {float((Y==IDX_S).mean()):.4f}")
_w = {k: v[1] - v[0] for k, v in SEGS22.items()}
if len(set(_w.values())) != 1:
    raise AssetError(f"폭이 다르다 {_w} — R27 ③ 위반")
_o = sorted(SEGS22.values())
for (a1, b1), (a2, b2) in zip(_o, _o[1:]):
    if b1 > a2:
        raise AssetError(f"창이 겹친다 {(a1,b1)} vs {(a2,b2)}")
for k_, (a_, b_) in SEGS22.items():
    run.log(f"    {k_:<14} index {a_:>3}–{b_:<3} (폭 {b_-a_})"
            f"  =  R{(a_-RPRE)/360*1000:+.0f}ms ~ R{(b_-RPRE)/360*1000:+.0f}ms")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【R-A】 특징 · 층 키 2종 · QRS 개시 · P 피크/PR 대용 · 점수
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def trend(pre_v, w):
    n = len(pre_v); out = np.zeros(n); x = np.arange(w, dtype=float)
    xc = x - x.mean(); den = float((xc * xc).sum())
    for i in range(n):
        a = i - w
        if a < 0:
            continue
        y = pre_v[a:i]
        out[i] = float(((y - y.mean()) * xc).sum() / den)
    return out

ALL_K = sorted(set(BASIS_K) | set(LEAK_K) | set(PROBE_K) | {HIST_K, KEY_ALT_K})

def all_feats(pre_v, post_v):
    med = float(np.median(pre_v)); n = len(pre_v)
    F = {"f1": med - pre_v}
    for k in ALL_K:
        F[f"f2_{k}"] = 1.0 - pre_v / np.maximum(local_base(pre_v, k), 1e-9)
    F["f6"] = 1.0 - local_base(pre_v, LB_K) / max(med, 1e-9)
    cv = np.empty(n)
    for i in range(n):
        a = max(0, i - TREND_W); w_ = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w_) / max(np.mean(w_), 1e-9))
    F["f4"] = cv
    F["trend"] = trend(pre_v, TREND_W)
    F["f5"] = np.r_[0.0, F[f"f2_{BASIS_K[0]}"][:-1]]
    F["f3"] = 1.0 - (pre_v + post_v) / np.maximum(2.0 * local_base(pre_v, BASIS_K[0]), 1e-9)
    return F

def basis_mat(F, kind):
    """`lin`/`rank` 는 기본 기저. ★ `ext` 는 **누출 채널을 직접 넣은 확장 기저**(dr 용)."""
    cols = ["f1"] + [f"f2_{k}" for k in BASIS_K] + ["f6"]
    if kind == "ext":
        cols = cols + [f"f2_{k}" for k in LEAK_K] + ["f4", "trend"]
    Z = np.stack([F[c] for c in cols], axis=1)
    if kind == "rank":
        Z = np.stack([stats.rankdata(F[c]) / len(F[c]) for c in cols], axis=1)
    elif kind not in ("lin", "ext"):
        raise AssetError(f"기저 종류 {kind} 를 모른다")
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-12)
    return np.c_[np.ones(len(Z)), Z]

def prep(s, kind):
    s = np.asarray(s, float)
    return stats.rankdata(s) / len(s) if kind == "rank" else s

def residualize(s, Z):
    s = np.asarray(s, float)
    if not np.isfinite(s).all():
        return None
    beta, *_ = np.linalg.lstsq(Z, s, rcond=None)
    e = s - Z @ beta
    return np.zeros_like(e) if float(np.std(e)) <= 1e-9 * (float(np.std(s)) + 1e-12) else e

def strat_key(F, k2=LB_K):
    """`f1` 정확값 × `f2_k` 구간. ★ `k2` 를 바꿔 R5(키 교체)를 한다."""
    pk = np.unique(np.round(F["f1"], 6), return_inverse=True)[1].astype(np.int64)
    f2b = np.floor(F[f"f2_{k2}"] / F2_BIN).astype(np.int64)
    return pk * (int(f2b.max() - f2b.min()) + 1) + (f2b - f2b.min())

def strat_auc(sc, tt, key):
    sc = np.asarray(sc, float)
    if not np.isfinite(sc).all():
        return float("nan")
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if not len(s_) or not len(n_):
            continue
        num += float((s_[:, None] > n_[None, :]).sum()) + 0.5 * float((s_[:, None] == n_[None, :]).sum())
        den += float(len(s_) * len(n_))
    return num / den if den >= 1 else float("nan")

def qrs_onset(B1):
    """3탭 평활 기울기로 QRS 개시. 반환 (index, **포화 여부**) — Q7-Q 픽스처가 잡은 결함."""
    lo, hi = ONSET_SEARCH
    d = np.abs(np.diff(B1, axis=-1))
    k = np.ones(3) / 3.0
    d = np.apply_along_axis(lambda v: np.convolve(v, k, mode="same"), 1, d)
    thr = ONSET_FRAC * (d[:, QRS_LO - 1:QRS_HI].max(axis=1) + 1e-12)
    out = np.full(len(B1), hi, dtype=np.int64); sat = np.ones(len(B1), bool)
    for i in range(len(B1)):
        cnt = 0
        for j in range(hi - 1, lo - 1, -1):
            if d[i, j] < thr[i]:
                cnt += 1
                if cnt >= ONSET_RUN:
                    out[i] = j + ONSET_RUN; sat[i] = False; break
            else:
                cnt = 0
    return np.clip(out, lo, hi), sat

def p_peak(Bm, seg):
    """`seg` 안에서 기저선 대비 |편차| 최대 지점(리드 0). PR 대용치의 재료."""
    a, b = seg
    w = Bm[:, 0, a:b].astype("float64")
    return a + np.argmax(np.abs(w - np.median(w, axis=1, keepdims=True)), axis=1)

def dist(Bw, ref):
    dd = Bw - ref[None]
    return np.sqrt((dd * dd).sum(axis=(1, 2)))

def two_template_cv(Bw, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            sc[te] = dist(Bw[te], np.median(Bw[tr & ~tt], axis=0)) \
                   - dist(Bw[te], np.median(Bw[tr & tt], axis=0))
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

RHY_COLS = ["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6"]   # 전부 RR 파생(창 특징 0개)

run.log("\n" + "=" * 100)
run.log("【R-A】 특징 · 층 키 · QRS 개시 · PR 대용")
run.log("=" * 100)
T0 = time.time()
FEAT, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append(int(r)); continue
    F = all_feats(PRE[mm], POST[mm]); Bm = BEAT[mm]
    on, sat = qrs_onset(Bm[:, 0, :])
    pk = p_peak(Bm, SEGS22[GATE_WIN])
    amp = float(np.median(Bm[:, :, QRS_LO:QRS_HI].max(axis=2)
                          - Bm[:, :, QRS_LO:QRS_HI].min(axis=2)))
    prv = np.r_[False, tt[:-1]]; nxt = np.r_[tt[1:], False]
    FEAT[int(r)] = F
    META[int(r)] = dict(tt=tt, idx=mm, amp=amp, onset=on, sat=float(sat.mean()),
                        pr=(on - pk).astype(float),          # ★ 스칼라 PR 대용치
                        key=strat_key(F), key_alt=strat_key(F, KEY_ALT_K),
                        iso=float((tt & ~prv & ~nxt).sum() / max(tt.sum(), 1)),
                        n=len(mm), pos=int(tt.sum()))
RS = sorted(META)
_sat = float(np.mean([META[r]["sat"] for r in RS]))
ONSET_OK = _sat < 0.5
_on = np.concatenate([META[r]["onset"] for r in RS])
_pr = np.concatenate([META[r]["pr"] for r in RS])
run.log(f"  채점 **{len(RS)}개체** · 제외 {len(SKIP)} · {time.time()-T0:.0f}초")
run.log(f"  QRS 개시 중앙 index {np.median(_on):.0f} (R{(np.median(_on)-RPRE)/360*1000:+.0f}ms)"
        f" · 포화율 {_sat:.3f}" + ("" if ONSET_OK else "  ⛔ **포화 — R4 개시정렬 미실행**"))
run.log(f"  PR 대용치(개시 − P피크) 중앙 {np.median(_pr):.1f}샘플 "
        f"({np.median(_pr)/360*1000:.0f}ms) · IQR [{np.percentile(_pr,25):.0f}, "
        f"{np.percentile(_pr,75):.0f}]")
run.log(f"  층 칸 수 — `f2_{LB_K}` 키 중앙 "
        f"{np.median([len(np.unique(META[r]['key'])) for r in RS]):.0f} · "
        f"`f2_{KEY_ALT_K}` 키 중앙 "
        f"{np.median([len(np.unique(META[r]['key_alt'])) for r in RS]):.0f}   ← R5 타당성")
CONFIG["onset_median"] = float(np.median(_on)); CONFIG["onset_sat"] = _sat
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【R-B】 점수 · 추정량 5종 채점 (raw · lin · rank · strat · dr)
ARMS_WIN = list(SEGS22)
PROBES = [f"f2_{k}" for k in PROBE_K]
LEAKC  = [f"f2_{k}" for k in LEAK_K]
EXTRA  = ["f1", "f2_16", "f3", "f4", "f5", "pr_scalar"]
LR_ARMS = ["lr_rhy"]
ARMS = EXTRA + PROBES + LEAKC + ARMS_WIN + LR_ARMS
LABEL_ARMS = ARMS_WIN + LR_ARMS

def win_arr(r, name, inject=None, shift=None):
    """창 파형. `inject=(amp, rng)` 면 **개체별 이질 파형**을 S 비트에 얹는다(R33 ①).
    `shift` 는 비트별 정수 이동(지터 대조·개시정렬용)."""
    a, b = SEGS22[name]; L = b - a
    Bm = BEAT[META[r]["idx"]]
    if shift is None:
        Bw = Bm[:, :, a:b].astype("float64").copy()
    else:
        Bw = np.stack([Bm[j, :, np.clip(a + shift[j], 0, Bm.shape[2] - L):
                             np.clip(a + shift[j], 0, Bm.shape[2] - L) + L]
                       for j in range(len(shift))]).astype("float64")
    if inject is not None:
        amp, rng = inject
        tt = META[r]["tt"]; x = np.arange(L, dtype=float)
        # ★ **개체마다 다른 파형** — 중심·폭·부호를 흔든다(동일 파형은 상한만 잰다)
        c = (L - 1) / 2.0 * (1.0 + HETERO * rng.uniform(-1, 1))
        w = max(L / 6.0 * (1.0 + HETERO * rng.uniform(-1, 1)), 1.0)
        sgn = 1.0 if rng.uniform() > 0.25 else -1.0
        Bw[tt] += sgn * amp * META[r]["amp"] * np.exp(-((x - c) ** 2) / (2 * w ** 2))[None, None, :]
    return Bw

def build_scores(r, tt, K, seed, inject=None):
    S = {}
    F = FEAT[r]
    X = np.stack([F[c] for c in RHY_COLS], axis=1)
    sc = cv_logit(X, tt, K, seed, N_REPEAT)
    if sc is None:
        return None
    S["lr_rhy"] = sc
    for nm_ in ARMS_WIN:
        inj = inject if (inject is not None and nm_ == GATE_WIN) else None
        st = two_template_cv(win_arr(r, nm_, inject=inj), tt, K, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

def score_of(r, a, S):
    if a in S:
        return S[a]
    return META[r]["pr"] if a == "pr_scalar" else FEAT[r][a]

run.log("\n" + "=" * 100)
run.log("【R-B】 추정량 5종 — " + " · ".join(LADDER))
run.log("=" * 100)
MET = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
SCORES = {}
T0 = time.time()
for i, r in enumerate(RS):
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    S = build_scores(r, tt, K_FOLD, SEED0)
    if S is None:
        continue
    SCORES[r] = S
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    for a in ARMS:
        s = score_of(r, a, S)
        if not np.isfinite(s).all():
            continue
        MET["raw"][a][i] = roc_auc_score(tt.astype(int), s)
        for b_ in ("lin", "rank"):
            e = residualize(prep(s, b_), Z[b_])
            if e is not None:
                MET[b_][a][i] = roc_auc_score(tt.astype(int), e)
        MET["strat"][a][i] = strat_auc(s, tt, key)
        # ★ dr — 확장 기저(누출 채널 포함) 잔차화 **뒤** 층화. 이중강건 **스타일**.
        e = residualize(s, Z["ext"])
        if e is not None:
            MET["dr"][a][i] = strat_auc(e, tt, key)
run.log(f"  ({time.time()-T0:.0f}초) 팔별 — " + " · ".join(LADDER))
for a in ARMS:
    star = "  ★" if a in ARMS_WIN else ("  ▸" if a in PROBES else "")
    run.log(f"    {a:<14} " + " · ".join(f"{np.nanmean(MET[m][a]):.4f}" for m in LADDER) + star)
CONFIG["metrics"] = {m: {a: float(np.nanmean(MET[m][a])) for a in ARMS} for m in LADDER}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【R-C】 라벨셔플 null (전 팔 · 추정량 5종 · R26 ②)
run.log("\n" + "=" * 100)
run.log(f"【R-C】 라벨셔플 null (셔플 {N_SHUF}회 · {len(RS)}개체)")
run.log("=" * 100)
T1 = time.time()
NULL = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
NSE  = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
for i, r in enumerate(RS):
    if r not in SCORES:
        continue
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    acc = {m: {a: [] for a in ARMS} for m in LADDER}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)
        Ss = build_scores(r, ts, K_FOLD, SEED0 + 31 * (s_ + 1))
        if Ss is None:
            continue
        for a in ARMS:
            s = score_of(r, a, Ss)
            if not np.isfinite(s).all():
                continue
            acc["raw"][a].append(roc_auc_score(ts.astype(int), s))
            for b_ in ("lin", "rank"):
                e = residualize(prep(s, b_), Z[b_])
                if e is not None:
                    acc[b_][a].append(roc_auc_score(ts.astype(int), e))
            acc["strat"][a].append(strat_auc(s, ts, key))
            e = residualize(s, Z["ext"])
            if e is not None:
                acc["dr"][a].append(strat_auc(e, ts, key))
    for m in LADDER:
        for a in ARMS:
            v_ = np.asarray([x for x in acc[m][a] if np.isfinite(x)], float)
            if len(v_) >= 2:
                NULL[m][a][i] = float(v_.mean())
                NSE[m][a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
run.log(f"  ({time.time()-T1:.0f}초)  초과분 (실측 − null)")
for a in ARMS:
    star = "  ★" if a in ARMS_WIN else ("  ▸" if a in PROBES else "")
    run.log(f"    {a:<14} " + " · ".join(
        f"{m} {np.nanmean(MET[m][a])-np.nanmean(NULL[m][a]):+.4f}" for m in LADDER) + star)
CONFIG["excess"] = {m: {a: float(np.nanmean(MET[m][a]) - np.nanmean(NULL[m][a]))
                        for a in ARMS} for m in LADDER}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【R-D】 ★★ R2 관문 통과형 양성 대조 + R3 추정량 선택(MDE)
def boot_pair(a1, a2, meth, seed, nb=NB_BOOT, q=2.5, arr=None):
    m1 = MET[meth][a1] if arr is None else arr
    d = (m1 - NULL[meth][a1]) - (MET[meth][a2] - NULL[meth][a2])
    se = np.sqrt(np.nan_to_num(NSE[meth][a1]) ** 2 + np.nan_to_num(NSE[meth][a2]) ** 2)
    ok = np.isfinite(d); d, se = d[ok], se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed); v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_excess(a, meth, seed, nb=NB_BOOT, q=2.5):
    d = MET[meth][a] - NULL[meth][a]; se = np.nan_to_num(NSE[meth][a])
    ok = np.isfinite(d); d, se = d[ok], se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed); v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log(f"【R-D】 ★★ R2 관문 통과형 양성 대조 (`{GATE_WIN}` 에 주입 · 개체별 이질 파형)")
run.log("=" * 100)
run.log("  ⚠️ Q7-Q 는 **수준 추정기**만 검정하고 관문의 검출력을 주장했다(R33 ①).")
run.log("     이번엔 주입을 **짝 관문에 그대로 통과**시킨다 — 같은 부트스트랩·같은 Bonferroni")
T2 = time.time()
INJ = {m: {a_: np.full(len(RS), np.nan) for a_ in AMPS} for m in LADDER}
for i, r in enumerate(RS):
    if r not in SCORES:
        continue
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    for a_ in AMPS:
        rng = np.random.RandomState(SEED0 + 977 * int(r) + int(a_ * 10000))
        st = two_template_cv(win_arr(r, GATE_WIN, inject=(a_, rng)), tt, K_FOLD,
                             SEED0, N_REPEAT)
        if st is None:
            continue
        INJ["raw"][a_][i] = roc_auc_score(tt.astype(int), st)
        for b_ in ("lin", "rank"):
            e = residualize(prep(st, b_), Z[b_])
            if e is not None:
                INJ[b_][a_][i] = roc_auc_score(tt.astype(int), e)
        INJ["strat"][a_][i] = strat_auc(st, tt, key)
        e = residualize(st, Z["ext"])
        if e is not None:
            INJ["dr"][a_][i] = strat_auc(e, tt, key)
run.log(f"  ({time.time()-T2:.0f}초)")

# ── ★ R3 — 추정량별 (회수량 × CI 폭 → MDE). MDE 최소를 다음 주 추정량으로.
run.log(f"\n  ★★ R3 추정량 선택 — 관문 `{GATE_WIN} − {NEG22}` 를 5종 추정량으로")
SEL, FLOOR = {}, {}
for m in LADDER:
    bm, blo, bhi, bn = boot_pair(GATE_WIN, NEG22, m, SEED0 + 11, q=BONF2 * 100)
    m_ = mde(blo, bhi)
    hit, row = None, []
    for a_ in AMPS:
        rm, rlo, rhi, _ = boot_pair(GATE_WIN, NEG22, m, SEED0 + 21, q=BONF2 * 100,
                                    arr=INJ[m][a_])
        det = rlo > 0
        row.append(f"a={a_:.3f} {rm-bm:+.4f}{'✅' if det else '❌'}")
        if hit is None and det:
            hit = a_
    SEL[m] = dict(base=bm, lo=blo, hi=bhi, n=bn, mde=m_, floor=hit)
    FLOOR[m] = hit
    run.log(f"    {m:<6} 기저 {bm:+.4f} [{blo:+.4f}, {bhi:+.4f}] · **MDE {m_:.4f}** · "
            f"바닥 {('a≥%.3f' % hit) if hit else '⛔ 미검출'}")
    run.log(f"           회수 — " + " | ".join(row))
_valid = [m for m in LADDER if np.isfinite(SEL[m]["mde"])]
BEST = min(_valid, key=lambda m: SEL[m]["mde"]) if _valid else None
# ★ R33 ② — 격자가 바닥을 감쌌나(최소점 미검출이 하나라도 있어야)
BRACKET = any(SEL[m]["floor"] is None or SEL[m]["floor"] > AMPS[0] for m in _valid)
run.log(f"\n    ★ **MDE 최소 추정량 = `{BEST}`** ({SEL[BEST]['mde']:.4f}) "
        f"— 다음 실행의 주 추정량으로 사전등록한다(R33 ③)" if BEST else "\n    ⛔ 선택 불가")
if not BRACKET:
    run.log(f"    ⛔ **격자가 바닥을 못 감쌌다** — 최소점 a={AMPS[0]} 도 전부 검출됐다.")
    run.log(f"       바닥은 **≤ {AMPS[0]} 미확정**이라고 적는다(R33 ②)")
else:
    run.log("    ✅ 격자가 바닥을 감쌌다 — 최소점에서 미검출이 나왔다(R33 ②)")
CONFIG["estimator_sel"] = {m: {k: (None if v is None else float(v))
                               for k, v in SEL[m].items()} for m in LADDER}
CONFIG["best_estimator"] = BEST; CONFIG["bracketed"] = bool(BRACKET)

# ── 누출 바닥 (양의 초과 최댓값 · R32 ④) + **음성 대조가 누출만 나르나** 확인
run.log(f"\n  누출 바닥 [{PRIMARY}] (양의 초과 최댓값 · R32 ④)")
pos, neg = {}, {}
for a in PROBES + LEAKC + ["f3", "f4", "f5"]:
    mm_, ll_, hh_, _ = boot_excess(a, PRIMARY, SEED0 + 15)
    (pos if mm_ > 0 else neg)[a] = mm_
    run.log(f"    {a:<9} 초과 {mm_:+.4f} [{ll_:+.4f}, {hh_:+.4f}]")
LEAK_MAX = max(pos.values(), default=0.0)
LEAK_ARM = max(pos, key=pos.get) if pos else "(없음)"
neg_ex = boot_excess(NEG22, PRIMARY, SEED0 + 16)[0]
run.log(f"    ★ 누출 바닥 **{LEAK_MAX:.4f}** (`{LEAK_ARM}`) — 수준 주장에만(R32 ⑤)")
run.log(f"    ★★ 음성 대조 `{NEG22}` 초과 **{neg_ex:+.4f}** vs 누출 바닥 {LEAK_MAX:.4f} "
        f"→ **고유항 {neg_ex - LEAK_MAX:+.4f}**")
run.log("       (0 에 가까우면 **음성 대조가 누출만 나른다** — Q7-Q 에서 +0.0012 였다)")
if neg:
    run.log("    ⚠️ 과잉보정 진단(바닥 아님) — " + " · ".join(f"{k} {v:+.4f}" for k, v in neg.items()))
CONFIG["leak_max"] = float(LEAK_MAX); CONFIG["neg_own"] = float(neg_ex - LEAK_MAX)
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【R-E】 ★★ R1 주 관문 — 교차환자 ΔAUPRC · 동작점 부분 AUC
# ⚠️ **천장효과 때문에 AUROC 가 아니다** — Q7-Q 에서 lr_rhy 무잔차 AUROC 가 0.9600 이라
#    그 위의 Δ 는 압축된다. 유병률 0.08 이면 **AUPRC · 고특이도 부분 AUC** 가 1차다(R33 ⑤).
def partial_auc(y, s, spec_lo):
    """특이도 [spec_lo, 1] 구간의 부분 AUC를 그 구간 넓이로 정규화(=동작점 지표)."""
    fpr, tpr, _ = roc_curve(y, s)
    hi = 1.0 - spec_lo
    m = fpr <= hi
    if m.sum() < 2:
        return float("nan")
    x, yv = fpr[m], tpr[m]
    if x[-1] < hi:
        x = np.r_[x, hi]; yv = np.r_[yv, np.interp(hi, fpr, tpr)]
    # ⚠️ `np.trapz` 는 numpy 2.x 에서 제거됐다(Colab 버전에 의존하면 안 된다) — 직접 계산
    area = float((np.diff(x) * (yv[:-1] + yv[1:]) / 2.0).sum())
    return area / hi

run.log("\n" + "=" * 100)
run.log(f"【R-E】 ★★ R1 주 관문 — 교차환자(LORO) 리듬 전용 vs +창 특징")
run.log("=" * 100)
run.log(f"  창 특징 = 창별 PCA 상위 {N_PCA}성분 (**학습 레코드에서만 적합** — 누수 없음)")
T3 = time.time()
RHY_ALL = np.concatenate([np.stack([FEAT[r][c] for c in RHY_COLS], 1) for r in RS])
WIN_ALL = {w: np.concatenate([BEAT[META[r]["idx"]][:, :, SEGS22[w][0]:SEGS22[w][1]]
                              .reshape(len(META[r]["idx"]), -1) for r in RS])
           for w in ARMS_WIN}
TT_ALL = np.concatenate([META[r]["tt"] for r in RS])
RID    = np.concatenate([np.full(META[r]["n"], r) for r in RS])
P_BASE = np.full(len(TT_ALL), np.nan)
P_FULL = {w: np.full(len(TT_ALL), np.nan) for w in ARMS_WIN}
for r in RS:                                   # ★ LORO — 개체 단위 분리
    te = RID == r; tr = ~te
    if int(TT_ALL[tr].sum()) < 20 or int(TT_ALL[te].sum()) < 1:
        continue
    mu, sd = RHY_ALL[tr].mean(0), RHY_ALL[tr].std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((RHY_ALL[tr] - mu) / sd, TT_ALL[tr].astype(int))
    P_BASE[te] = lr.decision_function((RHY_ALL[te] - mu) / sd)
    for w in ARMS_WIN:
        pca = PCA(n_components=N_PCA, random_state=SEED0).fit(WIN_ALL[w][tr])
        Xtr = np.c_[RHY_ALL[tr], pca.transform(WIN_ALL[w][tr])]
        Xte = np.c_[RHY_ALL[te], pca.transform(WIN_ALL[w][te])]
        mu2, sd2 = Xtr.mean(0), Xtr.std(0) + 1e-9
        lr2 = LogisticRegression(max_iter=3000, C=1.0)
        lr2.fit((Xtr - mu2) / sd2, TT_ALL[tr].astype(int))
        P_FULL[w][te] = lr2.decision_function((Xte - mu2) / sd2)
ok = np.isfinite(P_BASE)
run.log(f"  ({time.time()-T3:.0f}초) 채점 비트 {int(ok.sum()):,} · 레코드 {len(np.unique(RID[ok]))}")

def rec_boot_delta(fn, w, seed, nb=1200, q=2.5):
    """★ **레코드 단위 클러스터 부트스트랩** — 개체 간 의존을 정직하게 유지한다."""
    recs_ = np.unique(RID[ok])
    base = fn(TT_ALL[ok].astype(int), P_BASE[ok])
    full = fn(TT_ALL[ok].astype(int), P_FULL[w][ok])
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb):
        pick = rng.choice(recs_, len(recs_), replace=True)
        m = np.concatenate([np.where(RID[ok] == p)[0] for p in pick])
        y_ = TT_ALL[ok][m].astype(int)
        if y_.sum() < 5 or (1 - y_).sum() < 5:
            continue
        v.append(fn(y_, P_FULL[w][ok][m]) - fn(y_, P_BASE[ok][m]))
    v = np.asarray([x for x in v if np.isfinite(x)])
    if len(v) < 50:
        return float("nan"), float("nan"), float("nan"), base, full
    return (full - base, float(np.percentile(v, q)), float(np.percentile(v, 100 - q)),
            base, full)

VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

nrec = len(np.unique(RID[ok]))
for gname, fn, nm_ in (("R1", average_precision_score, "ΔAUPRC"),
                       ("R1b", lambda y, s: partial_auc(y, s, SPEC_LO),
                        f"Δ부분AUC(특이도≥{SPEC_LO})")):
    run.log(f"\n  {gname} — {nm_}")
    best_w, best = None, None
    for w in ARMS_WIN:
        d_, lo_, hi_, b_, f_ = rec_boot_delta(fn, w, SEED0 + 41, q=BONF2 * 100)
        star = "  ★" if w == GATE_WIN else ("  (음성)" if w == NEG22 else "")
        run.log(f"    {w:<14} 기저 {b_:.4f} → +창 {f_:.4f}  **Δ {d_:+.4f}** "
                f"[{lo_:+.4f}, {hi_:+.4f}]{star}")
        if w != NEG22 and (best is None or (np.isfinite(d_) and d_ > best[0])):
            best_w, best = w, (d_, lo_, hi_)
    if best is None or not np.isfinite(best[0]):
        g_(gname, "⛔ 측정 불가", "Δ 를 못 냈다"); continue
    d_, lo_, hi_ = best
    eq, sup, nn, m_, frame = judge(d_, lo_, hi_, nrec, EQ_DELTA)
    DIFF[gname] = dict(win=best_w, mean=d_, lo=lo_, hi=hi_, n=nrec, equiv=eq, sup=sup,
                       mde=float(m_), need_n=(None if nn is None else float(nn)))
    g_(gname, eq, f"★ 최량 창 `{best_w}` **Δ {d_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] "
               f"· 등가 여유 ±{EQ_DELTA} · Bonf 2 · 레코드 {nrec}")
    run.log(f"       우월성 {sup} · {frame}")
run.log(f"\n  ▸ 음성 대조 `{NEG22}` 의 Δ 가 P 창들과 같으면 **창 특이성이 없다**")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【R-F】 R4 타이밍 vs 형태 · 지터 대조 · R5 층 키 교체 (관문 아님)
run.log("\n" + "=" * 100)
run.log("【R-F】 R4 타이밍 vs 형태 · R5 층 키 (관문 아님)")
run.log("=" * 100)

# ── R4 (c) 스칼라 PR 단독 · (d) PR 잔차화
pr_m, pr_lo, pr_hi, _ = boot_excess("pr_scalar", PRIMARY, SEED0 + 51)
run.log(f"  R4(c) **스칼라 PR 대용치 단독** 초과 {pr_m:+.4f} [{pr_lo:+.4f}, {pr_hi:+.4f}]")
D_RESID = np.full(len(RS), np.nan)
for i, r in enumerate(RS):
    if r not in SCORES:
        continue
    tt = META[r]["tt"]; s = SCORES[r][GATE_WIN]
    if not np.isfinite(s).all():
        continue
    pr = META[r]["pr"]
    Zp = np.c_[np.ones(len(pr)), (pr - pr.mean()) / (pr.std() + 1e-9),
               stats.rankdata(pr) / len(pr)]
    e = residualize(s, Zp)
    if e is not None:
        D_RESID[i] = strat_auc(e, tt, META[r]["key"])
d_ = D_RESID - NULL[PRIMARY][GATE_WIN]
d_ = d_[np.isfinite(d_)]
base_lv = np.nanmean(MET[PRIMARY][GATE_WIN]) - np.nanmean(NULL[PRIMARY][GATE_WIN])
run.log(f"  R4(d) **PR 잔차화 후** `{GATE_WIN}` 초과 {d_.mean():+.4f} "
        f"(잔차화 전 {base_lv:+.4f} · 변화 {d_.mean()-base_lv:+.4f})")
run.log(f"        누출 바닥 {LEAK_MAX:.4f} 로 내려가면 **형태는 끝**이다")

# ── R4 (b) 개시정렬 + ★ 지터 대조 — Q7-Q 의 −0.0666 이 (A)타이밍인지 (B)잡음인지
if ONSET_OK:
    ALIGN = np.full(len(RS), np.nan); JIT = np.full(len(RS), np.nan)
    for i, r in enumerate(RS):
        if r not in SCORES:
            continue
        tt = META[r]["tt"]; on = META[r]["onset"]
        sh_align = (on - int(np.median(on))).astype(int)     # 개시에 정렬
        # ★ 지터 대조 — **정렬은 안 하고** 같은 크기의 무작위 이동만 준다
        rng = np.random.RandomState(SEED0 + 61 + int(r))
        sh_jit = rng.permutation(sh_align)
        for nm_, sh in (("align", sh_align), ("jit", sh_jit)):
            st = two_template_cv(win_arr(r, GATE_WIN, shift=sh), tt, K_FOLD,
                                 SEED0, N_REPEAT)
            if st is None:
                continue
            v = strat_auc(st, tt, META[r]["key"])
            (ALIGN if nm_ == "align" else JIT)[i] = v
    a_ex = np.nanmean(ALIGN) - np.nanmean(NULL[PRIMARY][GATE_WIN])
    j_ex = np.nanmean(JIT) - np.nanmean(NULL[PRIMARY][GATE_WIN])
    run.log(f"\n  R4(b) 고정 {base_lv:+.4f} · **개시정렬 {a_ex:+.4f}**({a_ex-base_lv:+.4f}) "
            f"· **지터만 {j_ex:+.4f}**({j_ex-base_lv:+.4f})")
    if np.isfinite(a_ex) and np.isfinite(j_ex):
        if abs(j_ex - a_ex) < 0.02:
            run.log("        → **(B) 검출기 잡음** — 정렬 없이 같은 크기 이동만 줘도 같은 낙폭")
        else:
            run.log("        → **(A) 타이밍 정보** — 정렬이 지터보다 더 깎는다. 고정 창은")
            run.log("           **QRS 개시 위치(=PR 간격)를 간접 부호화**하고 있었다")
    CONFIG["align"] = dict(fixed=float(base_lv), align=float(a_ex), jitter=float(j_ex))
else:
    run.log("\n  R4(b) ⛔ 검출기 포화로 미실행(R29 ②)")

# ── R5 층 키 교체 — 누출 바닥 모형 확증
run.log(f"\n  R5 층 키 `f2_{LB_K}` → `f2_{KEY_ALT_K}` (누출 채널을 키에 넣는다)")
ALT = {a: np.full(len(RS), np.nan) for a in (NEG22, GATE_WIN, f"f2_{LEAK_K[0]}")}
for i, r in enumerate(RS):
    if r not in SCORES:
        continue
    tt = META[r]["tt"]; ka = META[r]["key_alt"]
    for a in ALT:
        s = score_of(r, a, SCORES[r])
        if np.isfinite(s).all():
            ALT[a][i] = strat_auc(s, tt, ka)
for a in ALT:
    old = np.nanmean(MET[PRIMARY][a]) - np.nanmean(NULL[PRIMARY][a])
    run.log(f"    {a:<9} 초과 {old:+.4f} → (새 키 · null 미보정) 수준 "
            f"{np.nanmean(ALT[a]):.4f}  vs 기존 수준 {np.nanmean(MET[PRIMARY][a]):.4f}")
run.log(f"    ▸ `{NEG22}` 수준이 0.5 쪽으로 내려가면 **누출 바닥 모형이 확증**된다")
run.log("    ⚠️ 새 키의 null 은 재지 않았다 — **수준 비교만** 하고 초과분을 인용하지 않는다")
CONFIG["key_alt"] = {a: float(np.nanmean(ALT[a])) for a in ALT}
run.save_json("config", CONFIG)

In [ ]:
# CELL 9 — 【R-G】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))

# ① 추정량별 MDE (작을수록 좋다)
ms = [m for m in LADDER if np.isfinite(SEL[m]["mde"])]
ax[0].bar(range(len(ms)), [SEL[m]["mde"] for m in ms],
          color=["tab:green" if m == BEST else "tab:gray" for m in ms])
ax[0].set_xticks(range(len(ms))); ax[0].set_xticklabels(ms, rotation=30, fontsize=8)
ax[0].set_ylabel("MDE (CI half-width)"); ax[0].grid(alpha=.3, axis="y")
ax[0].set_xlabel("estimator  (lower is better)")

# ② 관문 통과형 회수 곡선
for m, c_ in zip(ms, ("tab:gray", "tab:blue", "tab:orange", "tab:red", "tab:green")):
    ys = [boot_pair(GATE_WIN, NEG22, m, SEED0 + 21, q=BONF2 * 100, arr=INJ[m][a_])[0]
          - SEL[m]["base"] for a_ in AMPS]
    ax[1].plot(AMPS, ys, "o-", color=c_, label=m)
ax[1].axhline(0, color="k", lw=.8)
for m in ms:
    ax[1].axhline(SEL[m]["mde"], ls=":", lw=.6, color="k")
ax[1].set_xscale("log")
ax[1].set_xlabel("injected amplitude (x median R)  [log]")
ax[1].set_ylabel("recovered pair difference")
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

# ③ R1 주 관문 CI + 등가 여유대
gs = [g for g in ("R1", "R1b") if g in DIFF]
ys = np.arange(len(gs))
if gs:
    mm_ = [DIFF[g]["mean"] for g in gs]
    lo_ = [DIFF[g]["lo"] for g in gs]; hi_ = [DIFF[g]["hi"] for g in gs]
    ax[2].errorbar(mm_, ys, xerr=[np.array(mm_) - np.array(lo_),
                                  np.array(hi_) - np.array(mm_)],
                   fmt="o", color="tab:blue", capsize=4)
    ax[2].set_yticks(ys)
    ax[2].set_yticklabels([f"{g}: {DIFF[g]['win']}" for g in gs], fontsize=8)
ax[2].axvspan(-EQ_DELTA, EQ_DELTA, color="tab:green", alpha=.15)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_xlabel(f"delta vs rhythm-only  (shaded = equivalence +-{EQ_DELTA})")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7r_calibrate", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")   # ★ 측정 불가는 어떤 분기도 안 탄다(R29 ②)

run.log(f"  ★ **추정량 선택(R3)** — MDE 최소 = `{BEST}` ({SEL[BEST]['mde']:.4f})"
        if BEST else "  ⛔ 추정량 선택 불가")
for m in LADDER:
    run.log(f"      {m:<6} MDE {SEL[m]['mde']:.4f} · 바닥 "
            f"{('a≥%.3f' % SEL[m]['floor']) if SEL[m]['floor'] else '미검출'}")
run.log(f"  ★ 격자가 바닥을 감쌌나 — {'✅ 감쌌다' if BRACKET else '⛔ **미확정**(R33 ②)'}")
run.log(f"  ★ 음성 대조 고유항 {CONFIG['neg_own']:+.4f} "
        "(0 에 가까우면 **누출만 나른다**는 Q7-Q 모형 재확인)")
run.log("")
for g in ("R1", "R1b"):
    d_ = DIFF.get(g)
    if d_ is None:
        run.log(f"  {g:<4}⛔ 측정 불가 — 어떤 결론 분기도 타지 않는다(R29 ②)"); continue
    run.log(f"  {g:<4}{VERD.get(g)} · 우월성 {d_['sup']} · Δ {d_['mean']:+.4f} "
            f"[{d_['lo']:+.4f}, {d_['hi']:+.4f}] · MDE {d_['mde']:.4f} (`{d_['win']}`)")
sup_ = lambda k: DIFF.get(k, {}).get("sup", "").startswith("✅")
below_mde = all(abs(DIFF[g]["mean"]) < DIFF[g]["mde"] for g in ("R1", "R1b") if g in DIFF)
if any(un_(g) or g not in DIFF for g in ("R1", "R1b")):
    run.log("\n  ⛔ 측정 불가가 있다 — 어떤 결론 분기도 타지 않는다(R29 ②)")
elif ok_("R1") and ok_("R1b"):
    run.log("\n  ★★ **형태 축 종료.** 창 형태는 리듬 위에 임상적으로 의미 있는 것을")
    run.log("     못 얹는다 — 전역 순위(AUPRC)와 동작점(부분 AUC) **둘 다** 등가다.")
    run.log("     퀘스트의 형태 갈래를 닫고 리듬·보정 쪽으로 간다")
elif ok_("R1") and sup_("R1b"):
    run.log("\n  ★★ **전역 순위는 안 바꾸는데 동작점은 바꾼다.** 가장 흥미로운 결과다 —")
    run.log("     `ecg-oppoint` 트랙으로 넘긴다")
elif sup_("R1") or sup_("R1b"):
    # ★ 등가 여유 밖 + CI 하한 > 0 → **창 형태가 실제로 기여한다**. 미결이 아니다(R31 ①).
    run.log("\n  ★★ **창 형태가 리듬 위에 실제로 기여한다** — 등가 여유 밖이고 CI 하한이")
    run.log("     0 을 뗀다. 등가 프레임을 버리고 **우월성 + 효과크기**로 읽는다(R31 ①).")
    run.log("     형태 축이 살아 있다 → 다음은 **Q7-O(교차환자 전이)**")
    for g in ("R1", "R1b"):
        if g in DIFF:
            run.log(f"       {g} Δ {DIFF[g]['mean']:+.4f} [{DIFF[g]['lo']:+.4f}, "
                    f"{DIFF[g]['hi']:+.4f}] (`{DIFF[g]['win']}`)")
elif below_mde:
    run.log("\n  ⛔ **효과가 MDE 아래다 — 측정 한계**(R33 ①). 「효과 없음」이 아니다.")
    run.log("     표본을 늘리거나(DB 풀링) 더 효율적인 추정량이 필요하다")
else:
    run.log("\n  ⚠️ 미결 — MDE 위인데 등가도 우월도 아니다. 필요 개체 수가 결과다(R30 ①)")
run.log(f"\n  ▸ 다음 실행의 주 추정량은 **`{BEST}`** 로 사전등록한다(R33 ③)")
run.log("  ▸ `dr` 는 이중강건 **스타일**이지 AIPW 가 아니다 — 진짜 AIPW 는 R1 의 비트")
run.log("    수준 전역 모형 쪽에서 정의된다(영향함수). 그 길은 이번에 열어두기만 했다")

run.finish({
    "exp_id": "quest46_q7r_calibrate",
    "metric": "svdb_delta_auprc_vs_rhythm",
    "value": float(DIFF.get("R1", {}).get("mean", float("nan"))),
    "passed": bool(ok_("R1") and ok_("R1b")),
    "summary": ("주 관문을 교차환자 ΔAUPRC 등가로 갈아타고, 양성 대조를 관문에 통과시켜 "
                "MDE 를 실측하고, 추정량을 회수량×CI폭으로 골랐다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "estimator_sel": CONFIG.get("estimator_sel", {}), "best_estimator": BEST,
    "bracketed": bool(BRACKET), "leak_max": CONFIG.get("leak_max"),
    "neg_own": CONFIG.get("neg_own"), "align": CONFIG.get("align", {}),
    "key_alt": CONFIG.get("key_alt", {}), "excess": CONFIG.get("excess", {}),
    "onset_median": CONFIG.get("onset_median"), "onset_sat": CONFIG.get("onset_sat"),
    "n_scored": len(RS), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-calibrate`")